In [1]:
import numpy as np
import pandas as pd
import time
from pulp import *

In [ ]:
EXP_ID = "EXP_test" 
alpha = 1.0     # Driver weight
beta = 2.0      # Penalty weight
N = 10          # Max avaible drivers
penalties = []  # Penalty coefficient list
matrix = 'mock_duty_matrix_100_500.csv' # Input matrix

In [2]:
def save_experiment_json(exp_id, prob, x, P, alpha, beta, n_limit, penalties, matrix_file):
    # 抓取求解資訊
    solve_time = getattr(prob, 'solutionTime', 0)
    solver_name = getattr(prob, 'usedSolver', "CBC")
    status = LpStatus[prob.status]
    
    # 決策結果
    selected_duties = [j for j, var in x.items() if value(var) > 0.5]
    cancelled_pows = [i for i, var in P.items() if value(var) > 0.5]

    # 將輸入與輸出封裝在一起
    experiment_data = {
        "metadata": {
            "exp_id": exp_id,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "matrix_used": matrix_file
        },
        "input_params": {
            "alpha": alpha,
            "beta": beta,
            "N_limit": n_limit,
            "penalties": penalties  # 雖然這串很長，但放在一起很保險
        },
        "output_results": {
            "status": status,
            "solver": solver_name,
            "runtime_sec": round(solve_time, 4),
            "total_cost": value(prob.objective) if status == 'Optimal' else None,
            "drivers_used": len(selected_duties),
            "cancelled_count": len(cancelled_pows),
            "selected_duties": selected_duties,
            "cancelled_pows": cancelled_pows
        }
    }

    with open(f"{exp_id}_full_report.json", "w", encoding="utf-8") as f:
        json.dump(experiment_data, f, indent=4)
    
    return experiment_data

In [3]:
# undone
def load_matrix(file_name):
    df = pd.read_csv(file_name, index_col=0)
    
    matrix_A = df.values
    
    m, n = matrix_A.shape
    
    return matrix_A, m, n

In [4]:
def build_and_solve_model(matrix_A, alpha, beta, n_limit, penalties):
    """
    Establish and solve the Set Covering model.
    """
    m, n = matrix_A.shape
    
    # --- 1. Initialize the model ---
    prob = LpProblem("Driver_Scheduling", LpMinimize)
    
    # --- 2. Define model variables ---
    x = LpVariable.dicts("Duty", range(n), cat='Binary')
    P = LpVariable.dicts("Penalty", range(m), cat='Binary')
    
    # --- 3. Objective function ---
    # min (alpha * sum x_j + beta * sum Pi * Ci)
    prob += (alpha * lpSum([x[j] for j in range(n)]) + 
             beta * lpSum([P[i] * penalties[i] for i in range(m)]))
    
    # --- 4. Constraints ---
    # A. Cover or Abandon: sum(x_ij) + Pi >= 1
    # Preprocessing: Find out which Duty j covers each PoW i to speed up modeling
    for i in range(m):
        covered_by_duties = [x[j] for j in range(n) if matrix_A[i][j] == 1]
        prob += lpSum(covered_by_duties) + P[i] >= 1
        
    # B. Max avaible drivers: sum(x_j) <= N
    prob += lpSum([x[j] for j in range(n)]) <= n_limit
    
    # --- 5. Solve ---
    # 如果有 Gurobi，可以換成 prob.solve(GUROBI_CMD())
    start_solve = time.time()
    prob.solve(PULP_CBC_CMD(msg=0))
    
    # 將求解耗時記錄在 prob 物件中供後續存檔使用
    prob.solutionTime = time.time() - start_solve
    
    return prob, x, P

In [ ]:
def run_experiment_pipeline(exp_id, alpha, beta, n_limit, matrix_file, penalties):
    # 1. 讀取數據 (使用你之前寫好的讀取函數)
    # 確保 matrix_A 是 numpy array 格式
    matrix_A, m, n = load_matrix(matrix_file)
    
    # 2. 建立並求解模型
    # build_and_solve_model 內部會記錄 prob.solutionTime 與 prob.usedSolver
    prob, x, P = build_and_solve_model(matrix_A, alpha, beta, n_limit, penalties)
    
    # 3. 呼叫整合後的存檔函數 (將參數與結果一次寫入)
    # 我們不需要在這邊手動算 runtime，因為 build_and_solve_model 已經算好存進 prob 了
    report = save_experiment_json(
        exp_id, prob, x, P, 
        alpha, beta, n_limit, penalties, matrix_file
    )
    
    # 4. 顯示簡要結果
    status = report['output_results']['status']
    total_cost = report['output_results']['total_cost']
    print(f"Experiment {exp_id} done！Status: {status}, Total cost: {total_cost}")